# Cellphi example

Segments organoids with Cellpose and measures each organoid's diameter (maximum Feret diameter) in µm.

**Google Colab:** Runtime → Change runtime type → GPU, then run the install cell below.

In [ ]:
# Colab only: install Cellphi (skip this cell if you installed it locally)
# !pip install -q git+<repository URL>

In [ ]:
import cv2
import matplotlib.pyplot as plt
from cellphi import Calibration, DEFAULT_CALIBRATION, annotate, process_image, organoid_rows, write_csv
from cellphi.segment import Segmenter

segmenter = Segmenter(gpu=True)  # falls back to CPU automatically
print("Running on", segmenter.device_name)

In [ ]:
# Calibration: 1800 µm = 370 px on a 926x695 export of the camera image = 2.20 µm/px on the
# full 2048x1536 images. Scaled automatically for other resolutions of the same field of view.
calibration = DEFAULT_CALIBRATION
# or e.g. Calibration.from_units_per_px(2.20)

image = "../input/Midbrain Organoids/Plate 1 CM_CM/A1/day 23 2-28-23 KS217537.jpg"
res = process_image(image, segmenter, calibration)
print(f"{len(res.measured)} organoids measured, {res.units_per_px:.3f} µm/px, "
      f"typical size used for Cellpose: {res.size_hint:.0f} µm")

In [ ]:
annotated = annotate(res.image, res.organoids, res.units_per_px)
plt.figure(figsize=(14, 10))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off");

In [ ]:
import statistics
complete = [d for d, o in zip(res.diameters(), res.measured) if not o.touches_border]  # skip edge-cut organoids
print(f"median {statistics.median(complete):.0f} µm, mean {statistics.fmean(complete):.0f} µm, sd {statistics.stdev(complete):.0f} µm")
plt.hist(complete, bins=20); plt.xlabel("diameter (µm)"); plt.ylabel("count");

In [ ]:
rows = organoid_rows(res.name, res.organoids, res.units_per_px, res.unit)
write_csv("organoids.csv", rows)
cv2.imwrite("annotated.jpg", annotated)